# PROJECT: Medical Insurance Cost Prediction.
# Business Goal: Help an insurance company predict annual charges accurately.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
import shap


print("1. Dear Batman, We are loading the dataset...")
df = pd.read_csv('insurance.csv')

# Debug: confirm columns match the attached file.
print("Columns found:", df.columns.tolist())
print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())

# Safety check for the problematic column
if 'smoker' not in df.columns:
    raise KeyError("Column 'Smoker' not found! Check your CSV File Header.")

In [ ]:
# Quick EDA.
print("\n2. Quick EDA")
print(df.describe())

# Target distribution 
plt.figure(figsize=(10, 6))
sns.histplot(df['charges'], kde=True, color='skyblue')
plt.title('Target Distribution (Annual charges)')
plt.xlabel('Charges ($)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.show()

# Correlation heatmap (numeric only)
plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# 3. Simple Baseline
print("\n3. Simple Baseline.")
baseline_mean = df['charges'].mean()
baseline_mae = mean_absolute_error(df['charges'], [baseline_mean] * len(df))
print(f"Baseline MAE (predict mean): ${baseline_mean:,.2f}")


In [ ]:
# 4. Feature engineering
print("\n4. Batman performed Feature Engineering")
# Fixed line: create numeric smoker column (fixed keyerror/typeerror)
df['smoker_num'] = (df['smoker'] == 'yes').astype(int)
print("Smoker_num created succesfully (0 = no, 1 = yes)")

# Critical FIX: Drop the original string column so LinearRegression doesnt break.
df = df.drop(columns=['smoker'])
print("Original 'smoker' column dropped(no more string dtype in features)")

# one hot encode the remaining categoricals.
df = pd.get_dummies(df, columns=['sex', 'region'], drop_first=True)

print("Final columns after encoding:", df.columns.tolist())
print(f"Final shape: {df.shape}")

# Prepare feature and target 
X = df.drop('charges', axis=1)
y = df['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape} | Test set: {X_test.shape}")

In [ ]:
# 5. Training Linear Regression Model
print("\n5. Here, Batman trained a linear Regression Model.")
bat_model = LinearRegression()
bat_model.fit(X_train, y_train)
preds = bat_model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"Linear Regression MAE: ${mae:,.2F}")
print(f"r2 score: {r2:.4f}")

In [ ]:
# SHAP INTERPRETABILITY
print("\n Batman is introducing SHAP feature importance.. be ready!")
explainer = shap.LinearExplainer(bat_model, X_train)

shap_values_array = explainer.shap_values(X_test)

# Fixed : Exlpicitly build Explanation with numpy arrays only 
# this forces proper dtypes so beeswarm's internal np.rint never sees a plain python float
shap_exp = shap.Explanation(
    values=shap_values_array.astype(float),
    base_values=np.full(len(X_test), float(explainer.expected_value)),  # Numpy array of floats
    data=X_test.values.astype(float),   # numpy array (no DataFrame)
    feature_names=X.columns.tolist()
)

print("Clean shap.Explanation object created with numpy arrays (no more rint errors)")
shap.plots.bar(shap_exp, max_display=10, show=True)
shap.plots.beeswarm(shap_exp, max_display=10, show=True)

In [ ]:
# 7. Model coefficients 
print("\n 7. Model Coefficients ")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': bat_model.coef_
}).sort_values(by='Coefficient', ascending=False)
print(coef_df.round(2))

In [ ]:
# Business Recommendations.
print("\n8. Business Recommendations.")
print("""
1. Smoking is the #1 driver of higher charges (last positive coefficients $ shap values.)
      Action: Introduce Smoker specific pricing tiers or strong wellness incentives.
2. Age and BMI strongly increase costs.
      Action: Age banded premiums + BMI based discounts for healthier members.
3. Number of children and certain regions (e.g south east) have moderate impact.
      Action: Family plan discounts and region adjusted risk pricing.
4. Business Value: Reduce under pricing risk on high cost members and gain edge on low risk profiles.
""")